# 1. Import Modules

In [ ]:
import csv
import json
import requests
import yaml
from datetime import datetime

# 2. Set Up & Authentication

In [ ]:
#Get the current date. This will be used to name the files created.
report_date = datetime.now().strftime("%m_%d_%Y")

#Safe loads the secrets file containing your username and password.
with open("../secrets.yml") as f:
    secrets = yaml.safe_load(f)

#Credentials to post for authentication
baseURL = 'https://archives.pratt.edu/staff/api'
user = secrets['username']
password = secrets['password']
repository = "2" #The Pratt Archives has only one repository which will always be "2"

#Sends the authentication request to the API
auth = requests.post(baseURL + '/users/' + user + '/login?password='+ password).json()

#If authentication fails, an error will be printed.
if 'session' not in auth:
    print("Error: authentication failed.")
    print(auth)
else:
    session = auth['session']
    headers = {'X-ArchivesSpace-Session': session,
            'Content_Type': 'application/json'}

    print('Authentication successful.')

# 3. Get Digital Objects and Dump Into JSON File

In [ ]:
#Endpoint for Get a list of Accessions for a Repository
endpoint = '/repositories/' + repository + '/digital_objects?all_ids=true'

#Get IDs for the accessions
ids = requests.get(baseURL + endpoint, headers=headers).json()
print(f"{len(ids)} digital objects found.")

#Iterate over IDs
records = []

print("Retrieving digital objects...")
counter = 1
for id in ids:
    endpoint = '/repositories/' + repository + '/digital_objects/' + str(id)
    output = requests.get(baseURL + endpoint, headers=headers).json()
    records.append(output)
    print(f"{counter}/{len(ids)}")

    counter +=1 

#Data is dumped into a JSON file
with open(f'../output/digital_objects_{report_date}.json', 'w', encoding='utf-8') as f:
    json.dump(records, f)

print(f"Completed. File created: ../output/digital_objects_{report_date}.json")

# 4. Create Column Headers for CSV File

In [ ]:
with open(f'../output/digital_objects_{report_date}.json', 'r', encoding='utf-8') as json_file:
    json_data = json.load(json_file)

#Basic field names
fieldnames = [
    'digital_object_id',
    'title',
    'publish',
    'restrictions',
    'level',
    'digital_object_type',
    'subjects',
    'linked_events',
    'external_documents',
    'rights_statements',
    'linked_agents',
    'notes',
    'uri'
]

#Create columns based on the maximum number of list items in each field
max_extents = max(len(r.get("extents", [])) for r in json_data)
max_dates = max(len(r.get("dates", [])) for r in json_data)
max_file_versions = max(len(r.get("file_versions", [])) for r in json_data)
max_collections = max(len(r.get("collection", [])) for r in json_data)
max_linked_instances = max(len(r.get("linked_instances", [])) for r in json_data)

for i in range(1, max_extents + 1):
    fieldnames.extend([
        f"extent_{i}_container_summary",
        f"extent_{i}_number",
        f"extent_{i}_portion",
       f"extent_{i}_extent_type"
    ])

for i in range(1, max_dates + 1):
    fieldnames.extend([
        f"date_{i}_begin",
        f"date_{i}_end",
        f"date_{i}_date_type",
        f"date_{i}_label"
    ])

for i in range(1, max_file_versions + 1):
    fieldnames.extend([
        f"file_version_{i}_file_uri",
        f"file_version_{i}_is_representative",
        f"file_version_{i}_caption",
        f"file_version_{i}_identifier",
    ])

for i in range(1, max_collections + 1):
    fieldnames.append(f"collection_{i}_ref")
    fieldnames.append(f"collection_{i}_title")

for i in range(1, max_linked_instances + 1):
    fieldnames.append(f"linked_instance_{i}_ref")
    fieldnames.append(f"linked_instance_{i}_name")

print(f"Columns to be added to CSV: {json.dumps(fieldnames, indent=2)}")

# 5. Write JSON to CSV
This writes a CSV file using the fieldnames listed above. Data for extents, dates, file versions, linked collections, and linked instances will be iterated out over sequential columns (example: date_1_begin, date_2_begin, date_3_begin, etc.). For readability, the script also retrieves the descriptive names for linked collections and archival objects. 

In [ ]:
with open(f'../output/digital_objects_{report_date}.json', 'r', encoding='utf-8') as json_file:
    json_data = json.load(json_file)

with open(f"../output/digital_objects_{report_date}.csv", "w", newline="", encoding="utf-8") as f:
    csv_writer = csv.DictWriter(f, fieldnames=fieldnames)
    csv_writer.writeheader()

    counter = 1
    for entry in json_data:

        row = {
            "digital_object_id": entry.get("digital_object_id"),
            "title": entry.get("title"),
            "publish": entry.get("publish"),
            "restrictions": entry.get("restrictions"),
            "level": entry.get("level"),
            "digital_object_type": entry.get("digital_object_type"),

            # Boolean whether list contains anything
            "subjects": bool(entry.get("subjects")),
            "linked_events": bool(entry.get("linked_events")),
            "external_documents": bool(entry.get("external_documents")),
            "rights_statements": bool(entry.get("rights_statements")),
            "linked_agents": bool(entry.get("linked_agents")),
            "notes": bool(entry.get("notes")),

            "uri": entry.get("uri"),
        }

        # Extents
        for i, extent in enumerate(entry.get("extents", []), start=1):
            row[f"extent_{i}_container_summary"] = extent.get("container_summary")
            row[f"extent_{i}_number"] = extent.get("number")
            row[f"extent_{i}_portion"] = extent.get("portion")
            row[f"extent_{i}_extent_type"] = extent.get("extent_type")

        # Dates
        for i, date in enumerate(entry.get("dates", []), start=1):
            row[f"date_{i}_begin"] = date.get("begin")
            row[f"date_{i}_end"] = date.get("end")
            row[f"date_{i}_date_type"] = date.get("date_type")
            row[f"date_{i}_label"] = date.get("label")

        # File Versions
        for i, fv in enumerate(entry.get("file_versions", []), start=1):
            row[f"file_version_{i}_file_uri"] = fv.get("file_uri")
            row[f"file_version_{i}_is_representative"] = fv.get("is_representative")
            row[f"file_version_{i}_caption"] = fv.get("caption")
            row[f"file_version_{i}_identifier"] = fv.get("identifier")

        # Collection
        for i, collection in enumerate(entry.get("collection", []), start=1):
            collection_ref = collection.get("ref")
            row[f"collection_{i}_ref"] = collection_ref

            response = requests.get(
                baseURL + collection_ref,
                headers=headers
            )

            if response.status_code == 200:
                response_json = response.json()
                row[f"collection_{i}_title"] = response_json.get("title")
            else:
                row[f"collection_{i}_title"] = ""

        # Linked Instances
        for i, instance in enumerate(entry.get("linked_instances", []), start=1):
            
            instance_ref = instance.get("ref")
            row[f"linked_instance_{i}_ref"] = instance_ref

            response = requests.get(
                baseURL + instance_ref,
                headers=headers
            )

            if response.status_code == 200:
                response_json = response.json()

                if instance_ref.startswith('/repositories/2/resources/'):
                    row[f"linked_instance_{i}_name"] = response_json.get("title")

                elif instance_ref.startswith("/repositories/2/archival_objects/"):
                    row[f"linked_instance_{i}_name"] = response_json.get("display_string")

                else: 
                    row[f"linked_instance_{i}_name"] = ""
            
            else:
                row[f"linked_instance_{i}_title"] = ""

        csv_writer.writerow(row)
        print(f"{counter}/{len(records)}")
        counter +=1
    
print(f"Completed. File created: ../output/digital_objects_{report_date}.csv")